<h1 style="font-size: 60px; font-weight: bold;text-align: center">Notebook AssurPrime</h1>

<h1 style="font-size: 20px; text-align: center">Théa El-Rostom - Agathe Guerrini-Touchais - Lucas Cabrera</h1>

# Import des bibliothèques 

In [1]:
import pandas as pd
import numpy as np

from category_encoders import CountEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import PoissonRegressor, TweedieRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import lightgbm as lgb

from sklearn.model_selection import GridSearchCV, KFold

from sklearn.metrics import make_scorer, mean_tweedie_deviance
from sklearn.metrics import mean_squared_error

# Modèle de référence de l'assureur

## Import des données

In [2]:
# Charger les données
print("Chargement des données...")
X_train = pd.read_csv("train_input.csv", low_memory=False)
y_train = pd.read_csv("train_output.csv")
X_test = pd.read_csv("test_input.csv", low_memory=False)
print("Données chargées avec succès.")

Chargement des données...
Données chargées avec succès.


## Traitement des données

In [3]:
# Traitement des valeurs manquantes dans les colonnes numériques
print("Traitement des valeurs manquantes dans les colonnes numériques...")
numeric_columns = X_train.select_dtypes(include=['number']).columns

# Remplir les NaN avec 0 pour les colonnes numériques
X_train[numeric_columns] = X_train[numeric_columns].fillna(0)

# Utilisation d'un Scaler
scaler = StandardScaler()
X_train[numeric_columns] = scaler.fit_transform(X_train[numeric_columns]) 

# Identifier les colonnes non numériques
fill_cols = [item for item in X_train.columns if item not in numeric_columns]

# Remplir les NaN des colonnes non numériques avec une valeur par défaut (-999)
X_train[fill_cols] = X_train[fill_cols].fillna(-999).astype(str)

print("Traitement des valeurs manquantes terminé.")

Traitement des valeurs manquantes dans les colonnes numériques...
Traitement des valeurs manquantes terminé.


On utilise un scaler afin d'optimiser le code avec l'algorithme neton cholesky (plus rapide)

In [4]:
# Préparation des données pour l'entraînement
print("Préparation des données pour l'entraînement...")

# Suppression des colonnes inutiles
X_train = X_train.drop(['ID', 'ANNEE_ASSURANCE'], axis=1)

# Encodage des variables catégoriques avec One-Hot Encoder
X_train_enc = pd.get_dummies(X_train)

#Remise à zéro des valeurs négatives
y_train['CM'] = y_train['CM'].clip(lower=0)

print("Préparation terminée.")

Préparation des données pour l'entraînement...
Préparation terminée.


Les valeurs négatives empêchent d'utiliser le GML avec Tweedie, on les enlèves donc de y_train

Au lieu d'utiliser un CountEncoder, avec lequel le code mettait du temps à tourner, Claude recommande un One-Hot Encoder pour optimiser le processus

## Entraînement des modèles

### Optimisation des Hyperparamètres

In [5]:
cv = KFold(n_splits=3, shuffle=True, random_state=42)

In [6]:
search_freq = GridSearchCV( PoissonRegressor(max_iter=1000000, solver="newton-cholesky"), 
                           param_grid={"alpha": [1e-3, 1e-2, 3e-2, 1e-1, 3e-1, 1]}, scoring="neg_mean_poisson_deviance",cv=cv)
search_freq.fit(X_train_enc, y_train["FREQ"])
print(search_freq.best_params_)

{'alpha': 0.01}


In [7]:
#Cellule suggérée par Claude car mon code search_cm avait planté (trop lourd)
rng = np.random.RandomState(0)
idx = rng.choice(len(X_train_enc), size=100_000, replace=False)
X_tune, y_tune = X_train_enc.iloc[idx], y_train.iloc[idx]

In [8]:
search_cm = GridSearchCV(  TweedieRegressor(link="log", max_iter=1000000, solver="newton-cholesky"),
                         param_grid={"alpha": [50, 75, 100, 125, 150], "power": [1.5]}, scoring="neg_root_mean_squared_error", cv=cv)
search_cm.fit(X_tune, y_tune["CM"])
print(search_cm.best_params_)

{'alpha': 125, 'power': 1.5}


Après une première recherche, alpha se trouve sur le bord du domaine ([1e-5, 1e-3, 1e-1] donne 0.1)et power au milieu ([1.3, 1.5, 1.7] donne 1.5), pour gagner du temps, je fixe power et teste d'autres alpha.

Pareil pour [0.1, 0.3, 1, 3, 10]

In [9]:
for s in (search_freq, search_cm):
    print(pd.DataFrame(s.cv_results_)[["params", "mean_test_score", "std_test_score"]]
          .sort_values("mean_test_score", ascending=False).to_string())

             params  mean_test_score  std_test_score
1   {'alpha': 0.01}        -0.122125        0.004777
2   {'alpha': 0.03}        -0.122468        0.005199
0  {'alpha': 0.001}        -0.122572        0.004700
3    {'alpha': 0.1}        -0.123516        0.005831
4    {'alpha': 0.3}        -0.125419        0.006589
5      {'alpha': 1}        -0.128088        0.007232
                         params  mean_test_score  std_test_score
3  {'alpha': 125, 'power': 1.5}     -6065.620302      355.830735
2  {'alpha': 100, 'power': 1.5}     -6065.627336      355.827206
4  {'alpha': 150, 'power': 1.5}     -6065.641406      355.832425
1   {'alpha': 75, 'power': 1.5}     -6065.696230      355.821191
0   {'alpha': 50, 'power': 1.5}     -6065.911638      355.809931


### Modèle GLM de Poisson --> Fréquence

In [10]:
model_freq = PoissonRegressor(alpha=0.01, max_iter=1000000,solver="newton-cholesky")  # alpha caractérise la pénalisation
model_freq.fit(X_train_enc, y_train['FREQ'])

print("Modèle pour 'FREQ' entraîné avec succès.")

Modèle pour 'FREQ' entraîné avec succès.


Le choix où non d'une pénalisation se fera dans la phase d'optimisation des HyperParamètres

### Modèle GLM de Tweedie --> CM

In [11]:
model_CM = TweedieRegressor(power=1.5, link='log', alpha=125, max_iter=1000000)
model_CM.fit(X_train_enc, y_train['CM'])

print("Modèle pour 'CM' entraîné avec succès.")

Modèle pour 'CM' entraîné avec succès.


### Prédictions sur l'ensemble d'entraînement

In [12]:
# Prédire 'FREQ' et 'CM' sur l'ensemble d'entraînement

y_train_pred_freq = model_freq.predict(X_train_enc)
y_train_pred_cm = model_CM.predict(X_train_enc)

In [13]:
# Calculer la prédiction combinée pour 'CHARGE'
y_train_pred = y_train_pred_freq * y_train_pred_cm * y_train['ANNEE_ASSURANCE']

In [14]:
# Calculer le RMSE sur l'ensemble d'entraînement
rmse = np.sqrt(mean_squared_error(y_train['CHARGE'], y_train_pred))
print(f"RMSE sur l'ensemble d'entraînement : {rmse:.2f}")

RMSE sur l'ensemble d'entraînement : 6802.77


## Prédictions sur le test

In [15]:
# Traitement des données de test
print("Traitement des données de test...")

# Remplir les valeurs manquantes
X_test[numeric_columns] = X_test[numeric_columns].fillna(0)
X_test[fill_cols] = X_test[fill_cols].fillna(-999).astype(str)

ids = X_test[["ID", "ANNEE_ASSURANCE"]].copy()

# Utilisation d'un Scaler
X_test[numeric_columns] = scaler.transform(X_test[numeric_columns])

# Suppression des colonnes inutiles
X_test_model = X_test.drop(['ID', 'ANNEE_ASSURANCE'], axis=1)

# Transformation des données de test avec le même encoder
X_test_enc = pd.get_dummies(X_test)

print("Traitement des données de test terminé.")

Traitement des données de test...
Traitement des données de test terminé.


In [16]:
# Prédictions sur l'ensemble de test
print("Prédictions sur l'ensemble de test...")

X_test_model = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)
y_pred_freq = model_freq.predict(X_test_model)
y_pred_CM = model_CM.predict(X_test_model)

resultats = ids.assign(FREQ_PRED=y_pred_freq)

Prédictions sur l'ensemble de test...


In [17]:
# Combiner les prédictions
y_pred = pd.concat([
    X_test[['ID', 'ANNEE_ASSURANCE']].reset_index(drop=True),
    pd.DataFrame(y_pred_freq, columns=['FREQ']),
    pd.DataFrame(y_pred_CM, columns=['CM'])
], axis=1)

In [18]:
# Calculer la colonne 'CHARGE' en multipliant 'FREQ', 'CM' et 'ANNEE_ASSURANCE'
y_pred['CHARGE'] = y_pred['CM'] * y_pred['FREQ'] * y_pred['ANNEE_ASSURANCE']

print("Prédictions terminées.")

Prédictions terminées.


In [19]:
# Exporter les prédictions dans un fichier CSV
print("Exportation des résultats...")
y_pred.to_csv('submission_Ref_V1.csv', index=False)
print("Fichier de soumission créé : 'submission_Ref_V1.csv'")

Exportation des résultats...
Fichier de soumission créé : 'submission_Ref_V1.csv'


# Modèle Exemple

## Traitement des données

In [20]:
# Charger les données
print("Chargement des données...")
X_train = pd.read_csv("train_input.csv", low_memory=False)
y_train = pd.read_csv("train_output.csv")
X_test = pd.read_csv("test_input.csv", low_memory=False)
print("Données chargées avec succès.")

Chargement des données...
Données chargées avec succès.


In [21]:
# Traitement des valeurs manquantes dans les colonnes numériques
print("Traitement des valeurs manquantes dans les colonnes numériques...")
numeric_columns = X_train.select_dtypes(include=['number']).columns

# Remplir les NaN avec 0 pour les colonnes numériques
X_train[numeric_columns] = X_train[numeric_columns].fillna(0)

# Identifier les colonnes non numériques
fill_cols = [item for item in X_train.columns if item not in numeric_columns]

# Remplir les NaN des colonnes non numériques avec une valeur par défaut (-999)
X_train[fill_cols] = X_train[fill_cols].fillna(-999)

print("Traitement des valeurs manquantes terminé.")

Traitement des valeurs manquantes dans les colonnes numériques...
Traitement des valeurs manquantes terminé.


In [22]:
# Préparation des données pour l'entraînement
print("Préparation des données pour l'entraînement...")

# Suppression des colonnes inutiles
X_train = X_train.drop(['ID', 'ANNEE_ASSURANCE'], axis=1)

# Encodage des variables catégoriques avec CountEncoder
encoder = CountEncoder(cols=fill_cols)
encoder.fit(X_train)
X_train_enc = encoder.transform(X_train)

print("Préparation terminée.")

Préparation des données pour l'entraînement...
Préparation terminée.


## Entraînement des modèles

### Modèle RandomForest --> Fréquence

In [23]:
# Entraîner les modèles
print("Entraînement des modèles")

# Modèle pour prédire 'FREQ'
rf_freq = RandomForestRegressor(random_state=42,n_jobs=-1)
rf_freq.fit(X_train_enc, y_train['FREQ'])
print("Modèle pour 'FREQ' entraîné avec succès.")

Entraînement des modèles
Modèle pour 'FREQ' entraîné avec succès.


### Modèle GradientBoostingRegressor --> CM

In [24]:
# Modèle pour prédire 'CM'
gb_cm = GradientBoostingRegressor(random_state=42)
gb_cm.fit(X_train_enc, y_train['CM'])
print("Modèle pour 'CM' entraîné avec succès.")

Modèle pour 'CM' entraîné avec succès.


### Prédictions sur l'ensemble d'entraînement

In [25]:
# Prédire 'FREQ' sur l'ensemble d'entraînement
y_train_pred_freq = rf_freq.predict(X_train_enc)

In [26]:
# Prédire 'CM' sur l'ensemble d'entraînement
y_train_pred_cm = gb_cm.predict(X_train_enc)

In [27]:
# Calculer la prédiction combinée pour 'CHARGE'
y_train_pred = y_train_pred_freq * y_train_pred_cm * y_train['ANNEE_ASSURANCE']

In [28]:
# Calculer le RMSE sur l'ensemble d'entraînement
rmse = np.sqrt(mean_squared_error(y_train['CHARGE'], y_train_pred))
print(f"RMSE sur l'ensemble d'entraînement : {rmse:.2f}")

RMSE sur l'ensemble d'entraînement : 6614.55


## Prédictions sur le test

In [29]:
# Traitement des données de test
print("Traitement des données de test...")

# Remplir les valeurs manquantes
X_test[numeric_columns] = X_test[numeric_columns].fillna(0)
X_test[fill_cols] = X_test[fill_cols].fillna(-999)

# Suppression des colonnes inutiles
X_test_model = X_test.drop(['ID', 'ANNEE_ASSURANCE'], axis=1)

# Transformation des données de test avec le même encoder
X_test_enc = encoder.transform(X_test_model)

print("Traitement des données de test terminé.")

Traitement des données de test...
Traitement des données de test terminé.


In [30]:
print("Prédictions sur l'ensemble de test...")

# Prédire 'FREQ' et 'CM' sur les données de test
y_pred_freq = rf_freq.predict(X_test_enc)
y_pred_cm = gb_cm.predict(X_test_enc)

Prédictions sur l'ensemble de test...


In [31]:
# Combiner les prédictions
y_pred = pd.concat([
    X_test[['ID', 'ANNEE_ASSURANCE']].reset_index(drop=True),
    pd.DataFrame(y_pred_freq, columns=['FREQ']),
    pd.DataFrame(y_pred_cm, columns=['CM'])
], axis=1)

In [32]:
# Calculer la colonne 'CHARGE' en multipliant 'FREQ', 'CM' et 'ANNEE_ASSURANCE'
y_pred['CHARGE'] = y_pred['CM'] * y_pred['FREQ'] * y_pred['ANNEE_ASSURANCE']

print("Prédictions terminées.")

Prédictions terminées.


In [33]:
# Exporter les prédictions dans un fichier CSV
print("Exportation des résultats...")
y_pred.to_csv('submission_EX.csv', index=False)
print("Fichier de soumission créé : 'submission_EX.csv'")

Exportation des résultats...
Fichier de soumission créé : 'submission_EX.csv'


# Modèles du cours appliqués à ce jeu de données

## Import et traitement des données

### Import

In [34]:
# Charger les données
print("Chargement des données...")
X_train = pd.read_csv("train_input.csv", low_memory=False)
Y_train = pd.read_csv("train_output.csv")
X_test = pd.read_csv("test_input.csv", low_memory=False)
print("Données chargées avec succès.")

Chargement des données...
Données chargées avec succès.


In [35]:
# Afficher un aperçu des données pour vérifier leur chargement
print("Aperçu de X_train :")
display(X_train.head())
print("Aperçu de y_train :")
display(Y_train.head())
print("Aperçu de X_test :")
display(X_test.head())

Aperçu de X_train :


,ID,ACTIVIT2,VOCATION,TYPERS,ANCIENNETE,ADOSS,CARACT1,CARACT2,CARACT3,INDEM1,...,NBJRR100_MMAX_A,NBJRR100_MSOM_A,RR_VOR_MM_A,RR_VOR_MMAX_A,RRAB_VOR_MM_A,RRAB_VOR_MMAX_A,ANNEE_ASSURANCE,ESPINSEE,AN_EXERC,ZONE
0,1,ACT1,VOC6,1,0,N,N,NaN,NaN,N,...,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,ANNEE5,3
1,2,ACT1,VOC6,1,0,N,N,NaN,NaN,N,...,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,ANNEE5,3
2,3,ACT1,VOC6,1,2,N,R,NaN,NaN,N,...,NaN,NaN,NaN,NaN,NaN,NaN,0.402740,NaN,ANNEE5,3
3,4,ACT1,VOC6,2,0,N,N,NaN,NaN,N,...,NaN,NaN,NaN,NaN,NaN,NaN,0.246575,NaN,ANNEE5,3
4,5,ACT1,VOC6,2,1,N,N,NaN,NaN,N,...,01. <= 0,01. <= 0,03. <= 102,03. <= 232,04. >= 25,02. <= 57,0.838356,NaN,ANNEE5,3


Aperçu de y_train :


,ID,FREQ,CM,ANNEE_ASSURANCE,CHARGE
0,1,0.0,0.0,1.000000,0.0
1,2,0.0,0.0,1.000000,0.0
2,3,0.0,0.0,0.402740,0.0
3,4,0.0,0.0,0.246575,0.0
4,5,0.0,0.0,0.838356,0.0


Aperçu de X_test :


,ID,ACTIVIT2,VOCATION,TYPERS,ANCIENNETE,ADOSS,CARACT1,CARACT2,CARACT3,INDEM1,...,NBJRR100_MMAX_A,NBJRR100_MSOM_A,RR_VOR_MM_A,RR_VOR_MMAX_A,RRAB_VOR_MM_A,RRAB_VOR_MMAX_A,ANNEE_ASSURANCE,ESPINSEE,AN_EXERC,ZONE
0,383611,ACT1,VOC6,2,4,N,R,NaN,NaN,N,...,01. <= 0,01. <= 0,03. <= 102,03. <= 232,03. <= 25,02. <= 57,0.813699,NaN,ANNEE5,3
1,383612,ACT1,VOC6,2,6,N,R,NaN,NaN,N,...,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,ANNEE5,3
2,383613,ACT1,VOC7,2,2,N,N,NaN,NaN,N,...,NaN,NaN,NaN,NaN,NaN,NaN,0.586301,NaN,ANNEE5,3
3,383614,ACT2,VOC1,2,2,N,N,NaN,NaN,N,...,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,ANNEE5,3
4,383615,ACT2,VOC1,2,3,N,R,NaN,NaN,N,...,01. <= 0,01. <= 0,02. <= 79,02. <= 176,03. <= 25,02. <= 57,0.753425,NaN,ANNEE5,3


In [36]:
X_train.describe()

,ID,TYPERS,ANCIENNETE,CARACT2,DUREE_REQANEUF,CARACT5,TYPBAT2,DEROG15,CA1,CA2,...,RISK5,RISK7,EQUIPEMENT1,EQUIPEMENT3,EQUIPEMENT4,EQUIPEMENT6,EQUIPEMENT7,ZONE_VENT,ANNEE_ASSURANCE,ZONE
count,383610.000000,383610.000000,383610.000000,15326.000000,357037.000000,383610.000000,383610.000000,383610.000000,383610.000000,383610.000000,...,357037.000000,357037.000000,383610.000000,383610.000000,383610.000000,383610.000000,383610.000000,365028.00000,383610.000000,383610.000000
mean,191805.500000,1.665368,5.175517,1.019053,2.200483,0.076515,1.061711,99.997263,1047.396444,4.105732,...,448.326103,92.964861,0.074740,0.029454,0.899468,5.905349,6.851302,1.92568,0.701557,49.046876
std,110738.812719,0.471862,3.823097,0.136714,1.300343,0.420928,0.831133,0.286544,4777.346410,329.381340,...,494.892793,8.902888,0.262972,0.566816,0.300708,4.290611,4.329113,0.62817,0.352375,27.919867
min,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,70.000000,0.000000,0.000000,...,0.000000,75.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.00000,0.002732,1.000000
25%,95903.250000,1.000000,2.000000,1.000000,2.000000,0.000000,0.000000,100.000000,0.000000,0.000000,...,4.000000,90.000000,0.000000,0.000000,1.000000,3.000000,4.000000,2.00000,0.389041,25.000000
50%,191805.500000,2.000000,5.000000,1.000000,2.000000,0.000000,1.000000,100.000000,0.000000,0.000000,...,5.000000,100.000000,0.000000,0.000000,1.000000,5.000000,6.000000,2.00000,0.882192,49.000000
75%,287707.750000,2.000000,9.000000,1.000000,2.000000,0.000000,2.000000,100.000000,0.000000,0.000000,...,1000.000000,100.000000,0.000000,0.000000,1.000000,8.000000,9.000000,2.00000,1.000000,73.000000
max,383610.000000,2.000000,11.000000,2.000000,10.000000,20.000000,2.000000,100.000000,30000.000000,30000.000000,...,1000.000000,500.000000,1.000000,87.000000,1.000000,21.000000,21.000000,3.00000,2.000000,95.000000


### Traitement

In [37]:
# Traitement des valeurs manquantes dans les colonnes numériques
print("Traitement des valeurs manquantes dans les colonnes numériques...")
numeric_columns = X_train.select_dtypes(include=['number']).columns

# Remplir les NaN avec 0 pour les colonnes numériques
X_train[numeric_columns] = X_train[numeric_columns].fillna(0)

# Identifier les colonnes non numériques
fill_cols = [item for item in X_train.columns if item not in numeric_columns]

# Remplir les NaN des colonnes non numériques avec une valeur par défaut (-999)
X_train[fill_cols] = X_train[fill_cols].fillna(-999)

print("Traitement des valeurs manquantes terminé.")

Traitement des valeurs manquantes dans les colonnes numériques...
Traitement des valeurs manquantes terminé.


In [38]:
# Préparation des données pour l'entraînement
print("Préparation des données pour l'entraînement...")

# Suppression des colonnes inutiles
X_train = X_train.drop(['ID', 'ANNEE_ASSURANCE'], axis=1)

# Encodage des variables catégoriques avec CountEncoder
encoder = CountEncoder(cols=fill_cols)
encoder.fit(X_train)
X_train_enc = encoder.transform(X_train)

print("Préparation terminée.")

Préparation des données pour l'entraînement...
Préparation terminée.


## Entrainement des modèles selon les données train

### Régression linéaire

In [39]:
#Importation des modèles
from sklearn.linear_model import LinearRegression

# Entraîner les modèles
print("Entraînement des modèles en régression linéaire")

# Modèle pour prédire 'FREQ'
reg=LinearRegression()
reg.fit(X_train_enc, Y_train['FREQ'])
print(reg.score(X_train_enc, Y_train['FREQ']))

print("Modèle de régression linéaire pour 'FREQ' entraîné avec succès.")


Entraînement des modèles en régression linéaire
0.002780301529376583
Modèle de régression linéaire pour 'FREQ' entraîné avec succès.


### Choix d'un échantillon dans les données d'entrainement

In [40]:
#Choix d'un échantillon dans les données

from sklearn.model_selection import train_test_split


X_tr, X_trtest, y_freq_tr, y_freq_trtest, y_cm_tr, y_cm_trtest, annee_tr, annee_trtest, charge_tr, charge_trtest = train_test_split(
    X_train_enc,
    Y_train['FREQ'],
    Y_train['CM'],
    Y_train['ANNEE_ASSURANCE'],
    Y_train['CHARGE'],
    test_size=0.2
)

### Prédictions sur l'ensemble d'entraînement selon plusieurs modèles

### RF - GB

In [41]:
# Entraîner les modèles
print("Entraînement des modèles")

# Modèle pour prédire 'FREQ'
rf = RandomForestRegressor(random_state=42,n_jobs=-1)
rf.fit(X_tr, y_freq_tr)
print("Modèle pour 'FREQ' entraîné avec succès.")

Entraînement des modèles
Modèle pour 'FREQ' entraîné avec succès.


In [42]:
# Modèle pour prédire 'CM'
gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_tr, y_cm_tr)
print("Modèle pour 'CM' entraîné avec succès.")

Modèle pour 'CM' entraîné avec succès.


In [43]:
# Prédire 'FREQ' sur l'échantillon d'entraînement
y_pred_freq_rf = rf.predict(X_tr)

# Prédire 'CM' sur l'échantillon d'entraînement
y_pred_cm_gb = gb.predict(X_tr)

In [44]:
## Calcul du RMSE sur l'échantillon de test

y_pred_freq_trtest = rf.predict(X_trtest)
y_pred_cm_trtest = gb.predict(X_trtest)

y_trtest_rf_gb = y_pred_freq_trtest * y_pred_cm_trtest * annee_trtest

RMSE_rf_gb = np.sqrt(mean_squared_error(charge_trtest, y_trtest_rf_gb))
print(f"RMSE RF-GB : {RMSE_rf_gb:.2f}")

RMSE RF-GB : 6665.19


In [45]:
# Calculer la prédiction combinée pour 'CHARGE'
# y_train_pred_rf_gb = y_pred_freq_rf * y_pred_cm_gb * annee_tr
# Calculer le RMSE sur l'ensemble d'entraînement
# rmse_rf_gb = np.sqrt(mean_squared_error(y_train['CHARGE'], y_train_pred_rf_gb))
# print(f"RMSE RF-GB : {rmse:.2f}")

### RF-RF

In [46]:
# Modèle pour prédire 'FREQ'
rf = RandomForestRegressor(random_state=42, n_jobs=-1)
rf.fit(X_tr, y_freq_tr)
print("Modèle pour 'FREQ' entraîné avec succès.")
# Prédire 'FREQ' sur l'ensemble d'entraînement
y_pred_freq_trtest = rf.predict(X_trtest)

# Modèle pour prédire 'CM'
rf.fit(X_tr, y_cm_tr)
print("Modèle pour 'CM' entraîné avec succès.")
y_pred_cm_trtest = rf.predict(X_trtest)

y_trtest_rf_rf = y_pred_freq_trtest * y_pred_cm_trtest * annee_trtest

Modèle pour 'FREQ' entraîné avec succès.
Modèle pour 'CM' entraîné avec succès.


In [47]:
RMSE_rf_rf = np.sqrt(mean_squared_error(charge_trtest, y_trtest_rf_rf))
print(f"RMSE RF-RF : {RMSE_rf_rf:.2f}")

RMSE RF-RF : 6988.64


### GB-GB

In [48]:
# Modèle pour prédire 'FREQ'
gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_tr, y_freq_tr)
print("Modèle pour 'FREQ' entraîné avec succès.")
# Prédire 'FREQ' sur l'ensemble d'entraînement
y_pred_freq_trtest = gb.predict(X_trtest)

# Modèle pour prédire 'CM'
gb.fit(X_tr, y_cm_tr)
print("Modèle pour 'CM' entraîné avec succès.")
y_pred_cm_trtest = gb.predict(X_trtest)

y_trtest_gb_gb = y_pred_freq_trtest * y_pred_cm_trtest * annee_trtest

RMSE_gb_gb = np.sqrt(mean_squared_error(charge_trtest, y_trtest_gb_gb))
print(f"RMSE GB-GB : {RMSE_gb_gb:.2f}")

Modèle pour 'FREQ' entraîné avec succès.
Modèle pour 'CM' entraîné avec succès.
RMSE GB-GB : 6662.10


### GB-RF

In [49]:

# Modèle pour prédire 'FREQ'
gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_tr, y_freq_tr)
print("Modèle pour 'FREQ' entraîné avec succès.")
# Modèle pour prédire 'CM'
rf = RandomForestRegressor(random_state=42, n_jobs=-1)
rf.fit(X_tr, y_cm_tr)
print("Modèle pour 'CM' entraîné avec succès.")

## Calcul du RMSE sur l'échantillon de test

y_pred_freq_trtest = gb.predict(X_trtest)
y_pred_cm_trtest = rf.predict(X_trtest)

y_trtest_gb_rf = y_pred_freq_trtest * y_pred_cm_trtest * annee_trtest

RMSE_gb_rf = np.sqrt(mean_squared_error(charge_trtest, y_trtest_gb_rf))
print(f"RMSE GB-RF : {RMSE_gb_rf:.2f}")

Modèle pour 'FREQ' entraîné avec succès.
Modèle pour 'CM' entraîné avec succès.
RMSE GB-RF : 6676.77


Le split 80/20 donne une première idée, mais sur un seul découpage les écarts (6904 contre 6905) ne sont pas significatifs. On confirme donc le choix de la combinaison par validation croisée.

### Choix de la combinaison par KFold

On évalue les 4 combinaisons (modèle FREQ, modèle CM) sur les mêmes plis, avec le RMSE sur CHARGE, et on retient celle qui **minimise** le RMSE moyen.

In [ ]:
from sklearn.base import clone

modeles = {"RF": RandomForestRegressor(random_state=42, n_jobs=-1),
           "GB": GradientBoostingRegressor(random_state=42)}
combinaisons = [("RF", "GB"), ("RF", "RF"), ("GB", "GB"), ("GB", "RF")]

kf = KFold(n_splits=3, shuffle=True, random_state=42)
resultats_kfold = []

for nom_freq, nom_cm in combinaisons:
    rmse_scores = []
    for train_index, val_index in kf.split(X_train_enc):
        X_tr, X_val = X_train_enc.iloc[train_index], X_train_enc.iloc[val_index]
        m_freq = clone(modeles[nom_freq]).fit(X_tr, Y_train['FREQ'].iloc[train_index])
        m_cm   = clone(modeles[nom_cm]).fit(X_tr, Y_train['CM'].iloc[train_index])
        y_pred_charge = m_freq.predict(X_val) * m_cm.predict(X_val) * Y_train['ANNEE_ASSURANCE'].iloc[val_index]
        rmse_scores.append(np.sqrt(mean_squared_error(Y_train['CHARGE'].iloc[val_index], y_pred_charge)))
    resultats_kfold.append({"FREQ": nom_freq, "CM": nom_cm,
                            "RMSE moyen": np.mean(rmse_scores), "ecart-type": np.std(rmse_scores)})
    print(f"{nom_freq}-{nom_cm} : {np.mean(rmse_scores):.2f} (+/- {np.std(rmse_scores):.2f})")

resultats_kfold = pd.DataFrame(resultats_kfold).sort_values("RMSE moyen")
meilleur_freq, meilleur_cm = resultats_kfold.iloc[0][["FREQ", "CM"]]
print(f"Combinaison retenue : {meilleur_freq} pour FREQ, {meilleur_cm} pour CM")
resultats_kfold

RF-GB : 6809.67 (+/- 79.62)
RF-RF : 7002.68 (+/- 48.56)
GB-GB : 6802.29 (+/- 85.94)
GB-RF : 6803.53 (+/- 85.81)
Combinaison retenue : GB pour FREQ, GB pour CM


,FREQ,CM,RMSE moyen,ecart-type
2,GB,GB,6802.292829,85.936508
3,GB,RF,6803.533978,85.810499
0,RF,GB,6809.671176,79.620606
1,RF,RF,7002.675685,48.562179


/opt/anaconda3/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Si deux combinaisons sont à moins d'un écart-type l'une de l'autre, la différence n'est pas significative.

### Optimisation des paramètres des modèles sélectionnés

On optimise ensuite par GridSearch les hyperparamètres des deux modèles de la combinaison retenue (le code s'adapte automatiquement à la combinaison choisie).

In [51]:
## FREQ
# CORRECTION : la grille est choisie selon le modèle retenu (avant : RF imposé pour FREQ)
my_kfold = KFold(n_splits=3, shuffle=True, random_state=42)

grilles = {"RF": {'n_estimators': [50, 100, 200], 'max_depth': [10, 20, None], 'min_samples_split': [2, 5]},
           "GB": {'n_estimators': [50, 100, 150], 'learning_rate': [0.01, 0.1, 0.2], 'max_depth': [3, 5]}}

grid_freq = GridSearchCV(clone(modeles[meilleur_freq]), grilles[meilleur_freq],
                         cv=my_kfold, scoring='neg_root_mean_squared_error', n_jobs=-1)
grid_freq.fit(X_train_enc, Y_train['FREQ'])
print(f"Meilleurs paramètres pour FREQ ({meilleur_freq}) :", grid_freq.best_params_)

Meilleurs paramètres pour FREQ (GB) : {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 50}


In [52]:
## CM
# CORRECTION : la grille est choisie selon le modèle retenu (avant : GB imposé pour CM)
grid_cm = GridSearchCV(clone(modeles[meilleur_cm]), grilles[meilleur_cm],
                       cv=my_kfold, scoring='neg_root_mean_squared_error', n_jobs=-1)
grid_cm.fit(X_train_enc, Y_train['CM'])
print(f"Meilleurs paramètres pour CM ({meilleur_cm}) :", grid_cm.best_params_)

Meilleurs paramètres pour CM (GB) : {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 50}


## Entraînement du meilleur modèle sur toutes les données d'entraînement

In [53]:
# CORRECTION : combinaison retenue + hyperparamètres du GridSearch
print("Entraînement des modèles")
freq_cours = clone(grid_freq.best_estimator_).fit(X_train_enc, Y_train['FREQ'])
print("Modèle pour 'FREQ' entraîné avec succès.")

Entraînement des modèles
Modèle pour 'FREQ' entraîné avec succès.


In [54]:
# Prédire 'FREQ' sur l'ensemble d'entraînement
y_train_pred_freq = freq_cours.predict(X_train_enc)

In [55]:
cm_cours = clone(grid_cm.best_estimator_).fit(X_train_enc, Y_train['CM'])
print("Modèle pour 'CM' entraîné avec succès.")

Modèle pour 'CM' entraîné avec succès.


In [56]:
# Prédire 'CM' sur l'ensemble d'entraînement
y_train_pred_cm = cm_cours.predict(X_train_enc)

In [57]:
# Calculer la prédiction combinée pour 'CHARGE'
y_train_pred = y_train_pred_freq * y_train_pred_cm * Y_train['ANNEE_ASSURANCE']

In [58]:
# RMSE sur l'ensemble d'entraînement : optimiste, NE PAS comparer aux autres modèles (voir le tableau du stacking)
rmse_train = np.sqrt(mean_squared_error(Y_train['CHARGE'], y_train_pred))
print(f"RMSE sur l'ensemble d'entraînement : {rmse_train:.2f}")

RMSE sur l'ensemble d'entraînement : 6802.42


## Tests

### Traitement des données de test

In [59]:
# Traitement des données de test
print("Traitement des données de test...")

# Remplir les valeurs manquantes
X_test[numeric_columns] = X_test[numeric_columns].fillna(0)
X_test[fill_cols] = X_test[fill_cols].fillna(-999)

# Suppression des colonnes inutiles
X_test_model = X_test.drop(['ID', 'ANNEE_ASSURANCE'], axis=1)

# Transformation des données de test avec le même encoder
X_test_enc = encoder.transform(X_test_model)

print("Traitement des données de test terminé.")

Traitement des données de test...
Traitement des données de test terminé.


### Prédictions sur les données de test

In [60]:
# Prédictions sur l'ensemble de test
print("Prédictions sur l'ensemble de test...")

# Prédire 'FREQ' et 'CM' sur les données de test
y_test_pred_freq_cours = freq_cours.predict(X_test_enc)
y_test_pred_cm_cours = cm_cours.predict(X_test_enc)

Prédictions sur l'ensemble de test...


In [61]:
# CORRECTION : on utilisait y_pred_freq / y_pred_cm (dernier pli du KFold) au lieu des prédictions sur le test
# Combiner les prédictions
y_pred = pd.concat([
    X_test[['ID', 'ANNEE_ASSURANCE']].reset_index(drop=True),
    pd.DataFrame(y_test_pred_freq_cours, columns=['FREQ']),
    pd.DataFrame(y_test_pred_cm_cours, columns=['CM'])
], axis=1)

In [62]:
# Calculer la colonne 'CHARGE' en multipliant 'FREQ', 'CM' et 'ANNEE_ASSURANCE'
y_pred['CHARGE'] = y_pred['CM'] * y_pred['FREQ'] * y_pred['ANNEE_ASSURANCE']

print("Prédictions terminées.")

Prédictions terminées.


In [63]:
# Exporter les prédictions dans un fichier CSV
print("Exportation des résultats...")
y_pred.to_csv('submission_Comparaison.csv', index=False)
print("Fichier de soumission créé : 'submission_Comparaison.csv'")

Exportation des résultats...
Fichier de soumission créé : 'submission_Comparaison.csv'


# Modèle LightGBM

## Import et traitement des données

### Import

In [92]:
# Charger les données
print("Chargement des données...")
X_train = pd.read_csv("train_input.csv", low_memory=False)
y_train = pd.read_csv("train_output.csv")
X_test = pd.read_csv("test_input.csv", low_memory=False)
print("Données chargées avec succès.")

Chargement des données...
Données chargées avec succès.


### Traitement

In [93]:
# Traitement des valeurs manquantes dans les colonnes numériques
print("Traitement des valeurs manquantes dans les colonnes numériques...")
numeric_columns = X_train.select_dtypes(include=['number']).columns

# Remplir les NaN avec 0 pour les colonnes numériques
X_train[numeric_columns] = X_train[numeric_columns].fillna(0)

# Identifier les colonnes non numériques
fill_cols = [item for item in X_train.columns if item not in numeric_columns]

# Remplir les NaN des colonnes non numériques avec une valeur par défaut 
# On choisit une chaine de caractères car la bibliothèque LightGBM gère nativement les données catégorielles textuelles
X_train[fill_cols] = X_train[fill_cols].fillna('Void')

print("Traitement des valeurs manquantes terminé.")

Traitement des valeurs manquantes dans les colonnes numériques...
Traitement des valeurs manquantes terminé.


In [94]:
# Préparation des données pour l'entraînement
print("Préparation des données pour l'entraînement...")

# Suppression des colonnes inutiles
X_train = X_train.drop(['ID', 'ANNEE_ASSURANCE'], axis=1)

# CORRECTION : on garde toutes les colonnes (avant : seulement les catégorielles) ; LightGBM gère nativement les catégories
X_train_enc = X_train.copy()
categories = {}
for c in fill_cols:
    X_train_enc[c] = X_train_enc[c].astype(str).astype('category')
    categories[c] = X_train_enc[c].cat.categories

#Remise à zéro des valeurs négatives
y_train['CM'] = y_train['CM'].clip(lower=0)

print("Préparation terminée.")

Préparation des données pour l'entraînement...
Préparation terminée.


L'encodage des variables catégorielles est traité nativement par le lightGBM

## Entraînement des modèles

### Optimisation des hyperparamètres

In [96]:
cv_charge = KFold(n_splits=5, shuffle=True, random_state=42)

In [69]:
param_grid_1 = { "n_estimators": [200, 500, 1000], "num_leaves": [8, 16, 31], "min_child_samples": [50, 100, 300]}

search_freq = GridSearchCV( lgb.LGBMRegressor(objective='poisson', random_state=42, learning_rate=0.01,verbose=-1),  param_grid=param_grid_1, scoring="neg_mean_poisson_deviance", cv=cv)

search_freq.fit(X_train_enc, y_train["FREQ"])
print(search_freq.best_params_)

{'min_child_samples': 300, 'n_estimators': 1000, 'num_leaves': 8}


In [97]:
from sklearn.model_selection import KFold

cv_charge = KFold(n_splits=5, shuffle=True, random_state=42)

# Quelques paramètres à comparer pour chaque cible
configs_freq = [
    {"learning_rate": 0.01, "n_estimators": 1000, "num_leaves": 8},
    {"learning_rate": 0.01, "n_estimators": 1000, "num_leaves": 16},
    {"learning_rate": 0.03, "n_estimators": 500,  "num_leaves": 16},
]

configs_cm = [
    {"learning_rate": 0.03, "n_estimators": 200, "num_leaves": 2},
    {"learning_rate": 0.03, "n_estimators": 500, "num_leaves": 4},
    {"learning_rate": 0.03, "n_estimators": 500, "num_leaves": 8},
]

y_freq = y_train["FREQ"].to_numpy()
y_cm = y_train["CM"].clip(lower=0).to_numpy()
y_charge = y_train["CHARGE"].to_numpy()
exposition = y_train["ANNEE_ASSURANCE"].to_numpy()

meilleur_rmse = float("inf")

# On teste les combinaisons de paramètres pour FREQ et CM
for params_freq in configs_freq:
    for params_cm in configs_cm:
        scores = []

        for train_idx, val_idx in cv_charge.split(X_train_enc):
            modele_freq = lgb.LGBMRegressor(
                objective="poisson", random_state=42, verbose=-1, **params_freq
            )
            modele_cm = lgb.LGBMRegressor(
                objective="tweedie", tweedie_variance_power=1.5,
                random_state=42, verbose=-1, **params_cm
            )

            modele_freq.fit(X_train_enc.iloc[train_idx], y_freq[train_idx])
            modele_cm.fit(X_train_enc.iloc[train_idx], y_cm[train_idx])

            pred_charge = (
                modele_freq.predict(X_train_enc.iloc[val_idx])
                * modele_cm.predict(X_train_enc.iloc[val_idx])
                * exposition[val_idx]
            )

            rmse = np.sqrt(mean_squared_error(y_charge[val_idx], pred_charge))
            scores.append(rmse)

        rmse_moyen = np.mean(scores)
        print(f"FREQ {params_freq} | CM {params_cm} | RMSE : {rmse_moyen:.2f}")

        if rmse_moyen < meilleur_rmse:
            meilleur_rmse = rmse_moyen
            best_params_freq = params_freq
            best_params_cm = params_cm

print("\\nMeilleur RMSE :", round(meilleur_rmse, 2))
print("Meilleurs paramètres FREQ :", best_params_freq)
print("Meilleurs paramètres CM :", best_params_cm)

FREQ {'learning_rate': 0.01, 'n_estimators': 1000, 'num_leaves': 8} | CM {'learning_rate': 0.03, 'n_estimators': 200, 'num_leaves': 2} | RMSE : 6794.40
FREQ {'learning_rate': 0.01, 'n_estimators': 1000, 'num_leaves': 8} | CM {'learning_rate': 0.03, 'n_estimators': 500, 'num_leaves': 4} | RMSE : 6794.43
FREQ {'learning_rate': 0.01, 'n_estimators': 1000, 'num_leaves': 8} | CM {'learning_rate': 0.03, 'n_estimators': 500, 'num_leaves': 8} | RMSE : 6794.47
FREQ {'learning_rate': 0.01, 'n_estimators': 1000, 'num_leaves': 16} | CM {'learning_rate': 0.03, 'n_estimators': 200, 'num_leaves': 2} | RMSE : 6794.40
FREQ {'learning_rate': 0.01, 'n_estimators': 1000, 'num_leaves': 16} | CM {'learning_rate': 0.03, 'n_estimators': 500, 'num_leaves': 4} | RMSE : 6794.43
FREQ {'learning_rate': 0.01, 'n_estimators': 1000, 'num_leaves': 16} | CM {'learning_rate': 0.03, 'n_estimators': 500, 'num_leaves': 8} | RMSE : 6794.48
FREQ {'learning_rate': 0.03, 'n_estimators': 500, 'num_leaves': 16} | CM {'learning_r

In [ ]:
param_grid_2 = { "reg_lambda": [0, 1, 10], "colsample_bytree": [0.5, 0.7, 1.0]}

search_freq = GridSearchCV( lgb.LGBMRegressor(objective='poisson', random_state=42, learning_rate=0.01,verbose=-1),  param_grid=param_grid_2, scoring="neg_mean_poisson_deviance", cv=cv)

search_freq.fit(X_train_enc, y_train["FREQ"])
print(search_freq.best_params_)

{'colsample_bytree': 0.5, 'reg_lambda': 10}


In [71]:
scorer_cm = make_scorer(mean_tweedie_deviance, power=1.5, greater_is_better=False)

param_grid_cm_1 = {"n_estimators": [50, 100, 200],"num_leaves": [2, 4, 8], "min_child_samples": [300, 600, 1000]}

search_cm = GridSearchCV(estimator=lgb.LGBMRegressor(objective='tweedie',tweedie_variance_power=1.5,random_state=42,learning_rate=0.03,verbose=-1),
                          param_grid=param_grid_cm_1,scoring=scorer_cm,cv=cv,n_jobs=-1)

search_cm.fit(X_train_enc, y_train["CM"])
print(search_cm.best_params_)

{'min_child_samples': 300, 'n_estimators': 200, 'num_leaves': 2}


In [72]:
param_grid_cm_2 = {"reg_lambda": [0,10, 20], "colsample_bytree": [0.3, 0.5, 0.7]}

search_cm_2 = GridSearchCV(estimator=lgb.LGBMRegressor(objective='tweedie',tweedie_variance_power=1.5,random_state=42,learning_rate=0.03,verbose=-1,min_child_samples=300,n_estimators=200,num_leaves=2),
    param_grid=param_grid_cm_2,scoring=scorer_cm,cv=cv)
search_cm_2.fit(X_train_enc, y_train["CM"])
print(search_cm_2.best_params_)
print(search_cm_2.best_score_)

{'colsample_bytree': 0.3, 'reg_lambda': 0}
-88.1134561196291


### Modèle lightGBM de Poisson --> Fréquence

In [100]:
model_freq_GBM = lgb.LGBMRegressor(objective='poisson',random_state=42, verbose=-1, n_jobs=-1, **best_params_freq) 
model_freq_GBM.fit(X_train_enc, y_train['FREQ'])

print("Modèle pour 'FREQ' entraîné avec succès.")

Modèle pour 'FREQ' entraîné avec succès.


Le choix où non d'une pénalisation se fera dans la phase d'optimisation des HyperParamètres

### Modèle lightGBM de Tweedie --> CM

In [101]:
# CORRECTION : learning_rate=0.03 comme dans le GridSearch (avec 0.01 et 200 arbres le modèle sous-apprend)
model_CM_GBM = lgb.LGBMRegressor(objective='tweedie',tweedie_variance_power=1.5,random_state=42,verbose=-1,**best_params_cm) 
model_CM_GBM.fit(X_train_enc, y_train['CM'])

print("Modèle pour 'CM' entraîné avec succès.")

Modèle pour 'CM' entraîné avec succès.


### Prédictions sur l'ensemble d'entraînement

In [102]:
# Prédire 'FREQ' et 'CM' sur l'ensemble d'entraînement

y_train_pred_freq_GBM = model_freq_GBM.predict(X_train_enc)
y_train_pred_cm_GBM = model_CM_GBM.predict(X_train_enc)

In [103]:
# Calculer la prédiction combinée pour 'CHARGE'
y_train_pred_GBM = y_train_pred_freq_GBM * y_train_pred_cm_GBM * y_train['ANNEE_ASSURANCE']

In [104]:
# Calculer le RMSE sur l'ensemble d'entraînement
rmse_GBM = np.sqrt(mean_squared_error(y_train['CHARGE'], y_train_pred_GBM))
print(f"RMSE sur l'ensemble d'entraînement : {rmse_GBM:.2f}")

RMSE sur l'ensemble d'entraînement : 6802.74


## Prédictions sur le test

In [105]:
# Traitement des données de test
print("Traitement des données de test...")

# Remplir les valeurs manquantes
X_test[numeric_columns] = X_test[numeric_columns].fillna(0)
X_test[fill_cols] = X_test[fill_cols].fillna('Void')


# Suppression des colonnes inutiles
ids = X_test[["ID", "ANNEE_ASSURANCE"]].copy()
X_test_model = X_test.drop(['ID', 'ANNEE_ASSURANCE'], axis=1)

# Transformation des données de test avec le même encoder
# CORRECTION : toutes les colonnes, avec les catégories du train (sinon les codes ne correspondent pas)
X_test_enc = X_test_model.copy()
for c in fill_cols:
    X_test_enc[c] = pd.Categorical(X_test_enc[c].astype(str), categories=categories[c])

print("Traitement des données de test terminé.")

Traitement des données de test...
Traitement des données de test terminé.


In [106]:
# Prédictions sur l'ensemble de test
print("Prédictions sur l'ensemble de test...")

X_test_model_GBM = X_test_enc[X_train_enc.columns]
y_pred_freq_GBM = model_freq_GBM.predict(X_test_model_GBM)
y_pred_CM_GBM = model_CM_GBM.predict(X_test_model_GBM)

resultats = ids.assign(FREQ_PRED=y_pred_freq_GBM)

Prédictions sur l'ensemble de test...


In [107]:
# Combiner les prédictions
y_pred_GBM = pd.concat([
    X_test[['ID', 'ANNEE_ASSURANCE']].reset_index(drop=True),
    pd.DataFrame(y_pred_freq_GBM, columns=['FREQ']),
    pd.DataFrame(y_pred_CM_GBM, columns=['CM'])
], axis=1)

In [108]:
# Calculer la colonne 'CHARGE' en multipliant 'FREQ', 'CM' et 'ANNEE_ASSURANCE'
y_pred_GBM['CHARGE'] = y_pred_GBM['CM'] * y_pred_GBM['FREQ'] * y_pred_GBM['ANNEE_ASSURANCE']

print("Prédictions terminées.")

Prédictions terminées.


In [109]:
# Exporter les prédictions dans un fichier CSV
print("Exportation des résultats...")
y_pred_GBM.to_csv('submission_GBM_V1.csv', index=False)
print("Fichier de soumission créé : 'submission_GBM_V1.csv'")

Exportation des résultats...
Fichier de soumission créé : 'submission_GBM_V1.csv'


# Stacking

Chaque couple (modèle FREQ, modèle CM) donne une prédiction de CHARGE = FREQ x CM x ANNEE_ASSURANCE. Le niveau 0 (prédictions out-of-fold des 3 modèles) est commun ; on compare ensuite **deux méthodes de niveau 1** :

- **Méthode 1 (cours)** : moindres carrés à poids positifs, erreur estimée par validation croisée K-Fold (K = 5).
- **Méthode 2 (nouvelle)** : régression Ridge dont le paramètre λ est choisi par **Leave-One-Out exact**, calculé en forme close (sans réentraîner n fois).

Les modèles de base sont des copies (`clone`) des modèles finaux des sections précédentes : il faut avoir exécuté tout le notebook avant cette section.

## Modèles de base

In [83]:
from sklearn.base import clone
from sklearn.model_selection import cross_val_predict
from sklearn.linear_model import LinearRegression

base_freq_cm = {
    "GLM Poisson / Tweedie":     (clone(model_freq),     clone(model_CM)),
    "Modèles du cours":          (clone(freq_cours),     clone(cm_cours)),
    "LightGBM Poisson / Tweedie": (clone(model_freq_GBM), clone(model_CM_GBM)),
}
for k, (f, c) in base_freq_cm.items():
    print(k, "|", f.get_params().get("alpha", ""), type(f).__name__, "/", type(c).__name__)

GLM Poisson / Tweedie | 0.01 PoissonRegressor / TweedieRegressor
Modèles du cours | 0.9 GradientBoostingRegressor / GradientBoostingRegressor
LightGBM Poisson / Tweedie |  LGBMRegressor / LGBMRegressor


## Données (un préprocessing par famille, identique à chaque section)

In [84]:
rmse = lambda y, p: np.sqrt(mean_squared_error(y, p))

X_train = pd.read_csv("train_input.csv", low_memory=False)
y_train = pd.read_csv("train_output.csv")
X_test  = pd.read_csv("test_input.csv",  low_memory=False)

ids_test = X_test[["ID", "ANNEE_ASSURANCE"]].copy()
annee_tr = y_train["ANNEE_ASSURANCE"].values
annee_te = X_test["ANNEE_ASSURANCE"].values
y_freq   = y_train["FREQ"].values
y_cm     = y_train["CM"].clip(lower=0).values
y_charge = y_train["CHARGE"].values

Xtr = X_train.drop(columns=["ID", "ANNEE_ASSURANCE"])
Xte = X_test.drop(columns=["ID", "ANNEE_ASSURANCE"])
num_cols = Xtr.select_dtypes(include="number").columns.tolist()
cat_cols = [c for c in Xtr.columns if c not in num_cols]
Xtr[num_cols] = Xtr[num_cols].fillna(0); Xte[num_cols] = Xte[num_cols].fillna(0)

# GLM : scaler + one-hot (comme le modèle de référence)
Xg_tr, Xg_te = Xtr.copy(), Xte.copy()
sc = StandardScaler().fit(Xg_tr[num_cols])
Xg_tr[num_cols] = sc.transform(Xg_tr[num_cols]); Xg_te[num_cols] = sc.transform(Xg_te[num_cols])
Xg_tr[cat_cols] = Xg_tr[cat_cols].fillna(-999).astype(str); Xg_te[cat_cols] = Xg_te[cat_cols].fillna(-999).astype(str)
Xg_tr = pd.get_dummies(Xg_tr)
Xg_te = pd.get_dummies(Xg_te).reindex(columns=Xg_tr.columns, fill_value=0)

# RF / GB : CountEncoder (comme les modèles du cours)
Xc_tr, Xc_te = Xtr.copy(), Xte.copy()
Xc_tr[cat_cols] = Xc_tr[cat_cols].fillna(-999); Xc_te[cat_cols] = Xc_te[cat_cols].fillna(-999)
enc = CountEncoder(cols=cat_cols).fit(Xc_tr)
Xc_tr, Xc_te = enc.transform(Xc_tr), enc.transform(Xc_te)

# LightGBM : catégories natives, mêmes catégories train / test
Xl_tr, Xl_te = Xtr.copy(), Xte.copy()
for c in cat_cols:
    Xl_tr[c] = Xl_tr[c].fillna('Void').astype(str).astype('category')
    Xl_te[c] = pd.Categorical(Xl_te[c].fillna('Void').astype(str), categories=Xl_tr[c].cat.categories)

donnees = {"GLM Poisson / Tweedie": (Xg_tr, Xg_te),
           "Modèles du cours": (Xc_tr, Xc_te),
           "LightGBM Poisson / Tweedie": (Xl_tr, Xl_te)}

## Niveau 0 : prédictions out-of-fold
Pour chaque pli, on entraîne sur les 4 autres et on prédit celui-ci : chaque ligne du train reçoit une prédiction faite par un modèle qui ne l'a jamais vue. Puis chaque modèle est réentraîné sur 100 % du train pour prédire le test.

Les modèles du cours (RF / GB) sont les plus longs : le commenter dans `base_freq_cm` pour un premier essai.

In [85]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
noms = list(base_freq_cm)
oof  = np.zeros((len(Xtr), len(noms)))
test = np.zeros((len(Xte), len(noms)))
rmse_plis = {n: [] for n in noms}
freq_cm_test = {}

for j, nom in enumerate(noms):
    m_f, m_c = base_freq_cm[nom]
    X, Xt = donnees[nom]
    for k, (tr, va) in enumerate(kf.split(X)):
        f = clone(m_f).fit(X.iloc[tr], y_freq[tr])
        c = clone(m_c).fit(X.iloc[tr], y_cm[tr])
        oof[va, j] = np.clip(f.predict(X.iloc[va]), 0, None) * np.clip(c.predict(X.iloc[va]), 0, None) * annee_tr[va]
        rmse_plis[nom].append(rmse(y_charge[va], oof[va, j]))
        print(f"{nom} - pli {k+1}/5 : RMSE {rmse_plis[nom][-1]:.2f}")
    f = clone(m_f).fit(X, y_freq); c = clone(m_c).fit(X, y_cm)
    pf, pc = np.clip(f.predict(Xt), 0, None), np.clip(c.predict(Xt), 0, None)
    freq_cm_test[nom] = (pf, pc)
    test[:, j] = pf * pc * annee_te

GLM Poisson / Tweedie - pli 1/5 : RMSE 7045.15
GLM Poisson / Tweedie - pli 2/5 : RMSE 6984.46
GLM Poisson / Tweedie - pli 3/5 : RMSE 7166.23
GLM Poisson / Tweedie - pli 4/5 : RMSE 6335.14
GLM Poisson / Tweedie - pli 5/5 : RMSE 6440.86
Modèles du cours - pli 1/5 : RMSE 7045.20
Modèles du cours - pli 2/5 : RMSE 6984.54
Modèles du cours - pli 3/5 : RMSE 7166.44
Modèles du cours - pli 4/5 : RMSE 6335.41
Modèles du cours - pli 5/5 : RMSE 6440.98
LightGBM Poisson / Tweedie - pli 1/5 : RMSE 7045.12
LightGBM Poisson / Tweedie - pli 2/5 : RMSE 6984.41
LightGBM Poisson / Tweedie - pli 3/5 : RMSE 7166.29
LightGBM Poisson / Tweedie - pli 4/5 : RMSE 6335.25
LightGBM Poisson / Tweedie - pli 5/5 : RMSE 6440.86


## Comparaison des modèles (RMSE out-of-fold, comparable entre modèles)

In [86]:
res = pd.DataFrame({
    "modele": noms,
    "RMSE OOF": [rmse(y_charge, oof[:, j]) for j in range(len(noms))],
    "ecart-type plis": [np.std(rmse_plis[n]) for n in noms],
})
res.loc[len(res)] = ["Constante (moyenne de CHARGE)", rmse(y_charge, np.full_like(y_charge, y_charge.mean())), np.nan]
res.loc[len(res)] = ["Moyenne simple des modèles", rmse(y_charge, oof.mean(axis=1)), np.nan]
res

,modele,RMSE OOF,ecart-type plis
0,GLM Poisson / Tweedie,6802.799546,338.574262
1,Modèles du cours,6802.943778,338.538762
2,LightGBM Poisson / Tweedie,6802.816111,338.546170
3,Constante (moyenne de CHARGE),6800.481464,NaN
4,Moyenne simple des modèles,6802.851296,NaN


## Méthode 1 : moindres carrés à poids positifs, validation K-Fold
Le méta-modèle cherche w ≥ 0 minimisant Σ (CHARGE_i − w·p_i)², où p_i est le vecteur des 3 prédictions out-of-fold du contrat i. Poids positifs et pas d'intercept : les poids se lisent comme la part de chaque modèle. L'erreur est estimée par K-Fold sur les prédictions OOF, comme pour les modèles de base.

In [87]:
meta1 = LinearRegression(positive=True, fit_intercept=False)
oof_stack1 = cross_val_predict(meta1, oof, y_charge, cv=KFold(5, shuffle=True, random_state=0))
meta1.fit(oof, y_charge)

rmse_m1 = rmse(y_charge, oof_stack1)
print("Poids méthode 1 :", dict(zip(noms, meta1.coef_.round(3))))
print(f"RMSE méthode 1 (K-Fold) : {rmse_m1:.2f}")

Poids méthode 1 : {'GLM Poisson / Tweedie': np.float64(3.129), 'Modèles du cours': np.float64(3.591), 'LightGBM Poisson / Tweedie': np.float64(40.748)}
RMSE méthode 1 (K-Fold) : 6798.50


## Méthode 2 : Ridge avec λ choisi par Leave-One-Out exact
**Pourquoi pas un LOO au niveau 0 ?** Il faudrait réentraîner chaque modèle de base n = 383 610 fois : impossible en temps de calcul (le cours le signale). Au niveau 1 en revanche, le méta-modèle est linéaire avec seulement 3 variables, et pour la Ridge l'erreur Leave-One-Out a une **forme close** :

$$\hat y_{-i} - y_i = \frac{\hat y_i - y_i}{1 - h_{ii}}, \qquad H = P\,(P^\top P + \lambda I)^{-1} P^\top$$

où P est la matrice n × 3 des prédictions OOF et h_ii la diagonale de H. Une seule résolution d'un système 3 × 3 donne donc les n erreurs LOO : on obtient le LOO exact (biais quasi nul) pour le coût d'un seul entraînement, et on s'en sert pour choisir λ.

La Ridge (vue en cours) stabilise les poids quand les prédictions des 3 modèles sont très corrélées, ce qui est le cas ici.

In [88]:
def ridge_loo(P, y, lam):
    # Ridge sans intercept + erreurs Leave-One-Out exactes en forme close
    A = P.T @ P + lam * np.eye(P.shape[1])
    w = np.linalg.solve(A, P.T @ y)
    h = np.einsum("ij,jk,ik->i", P, np.linalg.inv(A), P)   # diagonale de H, sans construire la matrice n x n
    err_loo = (P @ w - y) / (1 - h)
    return w, np.sqrt(np.mean(err_loo ** 2))

# grille de λ à l'échelle de P'P (les prédictions sont en euros)
echelle = np.trace(oof.T @ oof) / oof.shape[1]
grille = echelle * np.array([0, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1])

loo = pd.DataFrame([(l / echelle, ridge_loo(oof, y_charge, l)[1]) for l in grille],
                   columns=["lambda relatif", "RMSE LOO"])
print(loo.to_string(index=False))

lam_opt = grille[loo["RMSE LOO"].idxmin()]
w2, rmse_m2 = ridge_loo(oof, y_charge, lam_opt)
print("Poids méthode 2 :", dict(zip(noms, w2.round(3))))
print(f"RMSE méthode 2 (LOO exact) : {rmse_m2:.2f}")

 lambda relatif    RMSE LOO
       0.000000 6798.544642
       0.000001 6798.544641
       0.000010 6798.544635
       0.000100 6798.544577
       0.001000 6798.544005
       0.010000 6798.539593
       0.100000 6798.579311
       1.000000 6799.633841
Poids méthode 2 : {'GLM Poisson / Tweedie': np.float64(3.398), 'Modèles du cours': np.float64(3.735), 'LightGBM Poisson / Tweedie': np.float64(39.769)}
RMSE méthode 2 (LOO exact) : 6798.54


### Vérification de la formule
Sur un petit sous-échantillon, on compare la forme close au LOO « brut » (n réentraînements) : les deux doivent coïncider.

In [89]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import LeaveOneOut

idx = np.random.RandomState(0).choice(len(y_charge), 300, replace=False)
P_s, y_s = oof[idx], y_charge[idx]
lam_s = lam_opt * len(idx) / len(y_charge)

pred_brut = cross_val_predict(Ridge(alpha=lam_s, fit_intercept=False), P_s, y_s, cv=LeaveOneOut())
print(f"LOO brut (300 réentraînements) : {rmse(y_s, pred_brut):.4f}")
print(f"LOO forme close                : {ridge_loo(P_s, y_s, lam_s)[1]:.4f}")

LOO brut (300 réentraînements) : 29209.5256
LOO forme close                : 29209.5256


# Méthode d'optimisation Bayésienne

Afin d'optimiser au mieux les paramètres du lightGBM, qui est jusqu'à présent notre meilleur score, on essaie de lancer une optimisation bayésienne des hyperparamètres

In [ ]:
import optuna

cv = KFold(n_splits=5, shuffle=True, random_state=42)

y_freq = y_train["FREQ"].to_numpy()
y_cm = y_train["CM"].clip(lower=0).to_numpy()
y_charge = y_train["CHARGE"].to_numpy()
exposition = y_train["ANNEE_ASSURANCE"].to_numpy()

def objectif(trial):
    params_freq = {
        "learning_rate": trial.suggest_float("freq_lr", 0.005, 0.05, log=True),
        "n_estimators": trial.suggest_int("freq_n_estimators", 300, 1500),
        "num_leaves": trial.suggest_int("freq_num_leaves", 4, 32),
        "min_child_samples": trial.suggest_int("freq_min_child_samples", 100, 1000),
        "reg_lambda": trial.suggest_float("freq_reg_lambda", 0.1, 30, log=True),
        "colsample_bytree": trial.suggest_float("freq_colsample", 0.5, 1.0),
    }

    params_cm = {
        "learning_rate": trial.suggest_float("cm_lr", 0.005, 0.05, log=True),
        "n_estimators": trial.suggest_int("cm_n_estimators", 100, 1000),
        "num_leaves": trial.suggest_int("cm_num_leaves", 2, 16),
        "min_child_samples": trial.suggest_int("cm_min_child_samples", 100, 1500),
        "reg_lambda": trial.suggest_float("cm_reg_lambda", 0.1, 30, log=True),
        "colsample_bytree": trial.suggest_float("cm_colsample", 0.3, 1.0),
    }

    scores = []

    for train_idx, val_idx in cv.split(X_train_enc):
        modele_freq = lgb.LGBMRegressor( objective="poisson", random_state=42, verbosity=-1, n_jobs=-1, **params_freq )
        modele_cm = lgb.LGBMRegressor( objective="tweedie", tweedie_variance_power=1.5, random_state=42, verbosity=-1, n_jobs=-1, **params_cm )

        modele_freq.fit(X_train_enc.iloc[train_idx], y_freq[train_idx])
        modele_cm.fit(X_train_enc.iloc[train_idx], y_cm[train_idx])

        pred_charge = (modele_freq.predict(X_train_enc.iloc[val_idx]) * modele_cm.predict(X_train_enc.iloc[val_idx]) * exposition[val_idx] )

        scores.append(np.sqrt(mean_squared_error(y_charge[val_idx], pred_charge)))

    return np.mean(scores)

study = optuna.create_study(direction="minimize")
study.optimize(objectif, n_trials=10)

print("Meilleur RMSE :", study.best_value)
print("Meilleurs paramètres :", study.best_params)

[I 2026-11-02 17:36:15,759] A new study created in memory with name: no-name-9066018d-dba0-4960-a162-befea5997780
[I 2026-11-02 17:37:55,157] Trial 0 finished with value: 6794.408245279973 and parameters: {'freq_lr': 0.007989705707040719, 'freq_n_estimators': 470, 'freq_num_leaves': 10, 'freq_min_child_samples': 963, 'freq_reg_lambda': 5.960506803879214, 'freq_colsample': 0.5672703669393084, 'cm_lr': 0.02336277246122403, 'cm_n_estimators': 736, 'cm_num_leaves': 3, 'cm_min_child_samples': 610, 'cm_reg_lambda': 1.0274382079396889, 'cm_colsample': 0.6189535707046817}. Best is trial 0 with value: 6794.408245279973.
[I 2026-11-02 17:39:28,799] Trial 1 finished with value: 6794.456366418037 and parameters: {'freq_lr': 0.014446205442052199, 'freq_n_estimators': 502, 'freq_num_leaves': 23, 'freq_min_child_samples': 257, 'freq_reg_lambda': 2.4103018464804786, 'freq_colsample': 0.6391480383660026, 'cm_lr': 0.010313474289216067, 'cm_n_estimators': 849, 'cm_num_leaves': 13, 'cm_min_child_samples':

Meilleur RMSE : 6794.381071829189
Meilleurs paramètres : {'freq_lr': 0.03439475688556422, 'freq_n_estimators': 1095, 'freq_num_leaves': 10, 'freq_min_child_samples': 812, 'freq_reg_lambda': 0.35558966202245085, 'freq_colsample': 0.9509264878346767, 'cm_lr': 0.00884496820027774, 'cm_n_estimators': 453, 'cm_num_leaves': 7, 'cm_min_child_samples': 1140, 'cm_reg_lambda': 0.2552795645390696, 'cm_colsample': 0.8173156164762957}


Une fois les paramètres trouvés, on les injecte dans notre lightGBM

In [ ]:
# Récupérer les meilleurs paramètres Optuna et les convertir au format attendu par LightGBM
p = study.best_params

params_freq = {
    "learning_rate": p["freq_lr"],
    "n_estimators": p["freq_n_estimators"],
    "num_leaves": p["freq_num_leaves"],
    "min_child_samples": p["freq_min_child_samples"],
    "reg_lambda": p["freq_reg_lambda"],
    "colsample_bytree": p["freq_colsample"],
}

params_cm = {
    "learning_rate": p["cm_lr"],
    "n_estimators": p["cm_n_estimators"],
    "num_leaves": p["cm_num_leaves"],
    "min_child_samples": p["cm_min_child_samples"],
    "reg_lambda": p["cm_reg_lambda"],
    "colsample_bytree": p["cm_colsample"],
}

# Réentraîner les modèles sur toutes les données d'entraînement
modele_freq_final = lgb.LGBMRegressor( objective="poisson", random_state=42, verbosity=-1, n_jobs=-1, **params_freq )
modele_cm_final = lgb.LGBMRegressor( objective="tweedie", tweedie_variance_power=1.5, random_state=42, verbosity=-1, n_jobs=-1, **params_cm)

modele_freq_final.fit(X_train_enc, y_train["FREQ"])
modele_cm_final.fit(X_train_enc, y_train["CM"].clip(lower=0))

# Prédire sur le jeu de test
X_test_final = X_test_enc[X_train_enc.columns]

freq_pred = np.clip(modele_freq_final.predict(X_test_final), 0, None)
cm_pred = np.clip(modele_cm_final.predict(X_test_final), 0, None)
exposition_test = X_test["ANNEE_ASSURANCE"].to_numpy()

# Construire la soumission
submission = pd.DataFrame({
    "ID": X_test["ID"].to_numpy(),
    "ANNEE_ASSURANCE": exposition_test,
    "FREQ": freq_pred,
    "CM": cm_pred,
    "CHARGE": freq_pred * cm_pred * exposition_test,
})

submission.to_csv("submission_LGBM_Bayes.csv", index=False)
print("Fichier créé : submission_LGBM_Bayes.csv")
print(submission.head())

Fichier créé : submission_LGBM_Bayes.csv
       ID  ANNEE_ASSURANCE      FREQ          CM    CHARGE
0  383611         0.813699  0.025364  114.596407  2.365157
1  383612         1.000000  0.007468   55.266166  0.412708
2  383613         0.586301  0.003538   28.831199  0.059801
3  383614         1.000000  0.002398   70.523348  0.169131
4  383615         0.753425  0.003192  113.880221  0.273865


# Méthode de Lasso

In [115]:
# Charger les données
print("Chargement des données...")
X_train = pd.read_csv("train_input.csv", low_memory=False)
y_train = pd.read_csv("train_output.csv")
X_test = pd.read_csv("test_input.csv", low_memory=False)
print("Données chargées avec succès.")

Chargement des données...
Données chargées avec succès.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.pipeline import Pipeline
from sklearn.linear_model import Lasso

# 1. Extraction des données
X_lasso_train = X_train.drop(columns=["ID"]).copy()
y_lasso_train = y_train["CHARGE"].to_numpy()
X_lasso_test = X_test.drop(columns=["ID"]).copy()

num_cols_lasso = X_lasso_train.select_dtypes(include="number").columns
cat_cols_lasso = X_lasso_train.select_dtypes(exclude="number").columns

# 2. Pré-traitement optimisé (sparse_output=True pour éviter l'asphyxie de la RAM)
preparation_lasso = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), num_cols_lasso),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
    ]), cat_cols_lasso)
])

# 3. Configuration du pipeline
pipeline_lasso = Pipeline([
    ("prep", preparation_lasso),
    ("model", Lasso(max_iter=1000, tol=1e-2, random_state=42))
])

# 4. Recherche sur grille avec n_jobs=-1 pour tout paralléliser
recherche_lasso = GridSearchCV(
    pipeline_lasso,
    {"model__alpha": [1, 10, 100]},
    scoring="neg_root_mean_squared_error",
    cv=KFold(n_splits=3, shuffle=True, random_state=42),
    n_jobs=-1  # Utilise tous les cœurs disponibles
)

print("--- 1. Recherche du meilleur alpha ---")
recherche_lasso.fit(X_lasso_train, y_lasso_train)
best_alpha = recherche_lasso.best_params_["model__alpha"]
print(f"Meilleur alpha trouvé : {best_alpha}")
print(f"RMSE validation Lasso : {round(-recherche_lasso.best_score_, 2)}")

# 5. Entraînement final sur 100% des données d'entraînement
pipeline_lasso_final = Pipeline([
    ("prep", preparation_lasso),
    ("model", Lasso(alpha=best_alpha, max_iter=1000, tol=1e-2, random_state=42))
])

print("\n--- 2. Entraînement final sur tout le dataset ---")
pipeline_lasso_final.fit(X_lasso_train, y_lasso_train)

print("Prédiction sur le jeu de test...")
charge_pred_lasso = np.clip(pipeline_lasso_final.predict(X_lasso_test), 0, None)

# 6. Construction et export du fichier de soumission CSV
submission_lasso = X_test[["ID", "ANNEE_ASSURANCE"]].copy()

if 'freq_cm_test' in locals() and len(freq_cm_test) > 0:
    nom_ref = list(freq_cm_test.keys())[0]
    submission_lasso["FREQ"], submission_lasso["CM"] = freq_cm_test[nom_ref]
else:
    submission_lasso["FREQ"] = 0.05
    submission_lasso["CM"] = charge_pred_lasso / (submission_lasso["FREQ"] * submission_lasso["ANNEE_ASSURANCE"] + 1e-9)

submission_lasso["CHARGE"] = charge_pred_lasso

submission_filename = "submission_lasso_final.csv"
submission_lasso.to_csv(submission_filename, index=False)

print(f"\n--- 3. Succès ! Fichier créé : '{submission_filename}' ---")
display(submission_lasso.head())

--- 1. Recherche du meilleur alpha ---
Meilleur alpha trouvé : 10
RMSE validation Lasso : 6795.84

--- 2. Entraînement final sur tout le dataset ---
Prédiction sur le jeu de test...

--- 3. Succès ! Fichier créé : 'submission_lasso_final.csv' ---


,ID,ANNEE_ASSURANCE,FREQ,CM,CHARGE
0,383611,0.813699,0.05,7573.574972,308.130387
1,383612,1.000000,0.05,1854.887682,92.744386
2,383613,0.586301,0.05,214.103806,6.276468
3,383614,1.000000,0.05,2146.970099,107.348507
4,383615,0.753425,0.05,5627.285036,211.986771


## Comparaison des deux méthodes
Pour une comparaison à protocole égal, on calcule aussi l'erreur K-Fold de la méthode 2.

In [ ]:
from sklearn.linear_model import Ridge
oof_stack2_kf = cross_val_predict(Ridge(alpha=lam_opt, fit_intercept=False), oof, y_charge,
                                  cv=KFold(5, shuffle=True, random_state=0))
comparaison = pd.DataFrame({
    "methode": ["Meilleur modèle seul", "Moyenne simple", "Méthode 1 : moindres carrés positifs",
                "Méthode 2 : Ridge, λ par LOO"],
    "RMSE K-Fold": [res.iloc[:len(noms)]["RMSE OOF"].min(), rmse(y_charge, oof.mean(axis=1)),
                    rmse_m1, rmse(y_charge, oof_stack2_kf)],
    "RMSE LOO": [np.nan, np.nan, np.nan, rmse_m2],
})
comparaison

## Soumissions
Un fichier par méthode. CHARGE = prédiction stackée ; FREQ et CM repris du meilleur modèle seul (format attendu : 5 colonnes).

In [91]:
meilleur = res.iloc[:len(noms)].sort_values("RMSE OOF")["modele"].iloc[0]
for nom_fichier, pred in [("submission_stacking_M1.csv", meta1.predict(test)),
                          ("submission_stacking_M2.csv", test @ w2)]:
    sub = ids_test.copy()
    sub["FREQ"], sub["CM"] = freq_cm_test[meilleur]
    sub["CHARGE"] = np.clip(pred, 0, None)
    sub.to_csv(nom_fichier, index=False)
    print(f"Fichier de soumission créé : '{nom_fichier}'")

Fichier de soumission créé : 'submission_stacking_M1.csv'
Fichier de soumission créé : 'submission_stacking_M2.csv'
